In [1]:

# =============================================================================
# Section 1: Build C++ optimizer and run on all datasets
# =============================================================================
import subprocess, os, json

PROJ_ROOT = os.path.dirname(os.getcwd())          # .../symforce/
REPO_ROOT = os.path.dirname(os.path.dirname(PROJ_ROOT))  # workspace root
BUILD_DIR = os.path.join(PROJ_ROOT, "build")
BIN       = os.path.join(BUILD_DIR, "pose_graph_optimizer")
DATA_DIR  = os.path.join(REPO_ROOT, "data", "iSAM2")

DATASETS_TO_RUN = [
    "city10000.txt",
    "cityTrees10000.txt",
    "manhattanOlson3500.txt",
    "sphere400.txt",
    "sphere2500.txt",
    "torus10000.txt",
    "torus2000Points.txt",
]

# ---- Build ------------------------------------------------------------------
makefile = os.path.join(BUILD_DIR, "Makefile")
if not os.path.exists(makefile):
    print("Configuring CMake …")
    r = subprocess.run(
        ["cmake", "-B", BUILD_DIR, PROJ_ROOT, "-DCMAKE_BUILD_TYPE=Release"],
        cwd=PROJ_ROOT, capture_output=True, text=True
    )
    if r.returncode != 0:
        print("CMAKE CONFIGURATION FAILED:\n", r.stderr[-2000:])
        raise RuntimeError("cmake configure failed")
    print("  configured OK")

print("Building pose_graph_optimizer …")
r = subprocess.run(
    ["cmake", "--build", BUILD_DIR, "--parallel", "--target", "pose_graph_optimizer"],
    cwd=PROJ_ROOT, capture_output=True, text=True
)
if r.returncode != 0:
    print("BUILD FAILED:\n", r.stderr[-2000:])
    raise RuntimeError("cmake build failed")
print("  build OK ✓\n")

# ---- Run optimizer on each dataset -----------------------------------------
RESULTS = {}   # stem -> dict loaded from JSON

for dat_file in DATASETS_TO_RUN:
    dat_path  = os.path.join(DATA_DIR, dat_file)
    stem      = os.path.splitext(dat_file)[0]
    json_path = os.path.join(BUILD_DIR, f"{stem}.json")

    if not os.path.exists(dat_path):
        print(f"  [SKIP] {dat_file} — data file not found")
        continue

    print(f"  {stem} …", flush=True)
    r = subprocess.run([BIN, dat_path, BUILD_DIR], capture_output=True, text=True)

    if r.returncode != 0:
        print("FAILED")
        print((r.stderr + r.stdout)[-500:])
        continue

    if not os.path.exists(json_path):
        print("FAILED -- JSON not created")
        continue

    with open(json_path) as jf:
        res = json.load(jf)

    RESULTS[stem] = res
    pct = (res['initial_error'] - res['final_error']) / max(res['initial_error'], 1e-30) * 100
    status = "✓" if pct > 50 else "!"
    print(f"{status}  poses={res['num_poses']}  "
          f"err {res['initial_error']:.3e} → {res['final_error']:.3e}  "
          f"({pct:.1f}% reduction)  iters={res['num_iterations']}")

print("\nDone. Results available in RESULTS dict.")


Configuring CMake …


  configured OK
Building pose_graph_optimizer …
  build OK ✓

  city10000 …
✓  poses=10000  err 3.271e+08 → 2.560e+02  (100.0% reduction)  iters=116
  cityTrees10000 …
✓  poses=10000  err 5.154e+05 → 2.723e+02  (99.9% reduction)  iters=39
  manhattanOlson3500 …
✓  poses=3500  err 1.283e+06 → 7.304e+01  (100.0% reduction)  iters=52
  sphere400 …
✓  poses=400  err 4.272e+05 → 5.366e+01  (100.0% reduction)  iters=29
  sphere2500 …
✓  poses=2500  err 1.596e+06 → 3.651e+02  (100.0% reduction)  iters=52
  torus10000 …
✓  poses=10000  err 6.651e+06 → 3.324e+03  (100.0% reduction)  iters=13
  torus2000Points …
✓  poses=1073  err 4.490e+05 → 1.313e+03  (99.7% reduction)  iters=11

Done. Results available in RESULTS dict.


In [2]:

# =============================================================================
# Section 2: Parsing helpers and ground-truth reconstruction
# =============================================================================
import json, math, os
from collections import defaultdict, deque
import matplotlib
matplotlib.use('inline')
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

# Re-derive paths if this cell is run independently
if 'PROJ_ROOT' not in dir():
    PROJ_ROOT = os.path.dirname(os.getcwd())
    REPO_ROOT = os.path.dirname(os.path.dirname(PROJ_ROOT))

# ---- SE2 helpers ------------------------------------------------------------
def parse_se2_file(path):
    """Return (edges, landmarks) from EDGE2/ODOMETRY/LANDMARK file."""
    edges, landmarks = [], {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            tok = parts[0]
            if tok in ("EDGE2", "ODOMETRY"):
                edges.append((int(parts[1]), int(parts[2]),
                               float(parts[3]), float(parts[4]), float(parts[5])))
            elif tok == "LANDMARK":
                pid, lid = int(parts[1]), int(parts[2])
                dx, dy = float(parts[3]), float(parts[4])
                if lid not in landmarks:
                    landmarks[lid] = (pid, dx, dy)
    return edges, landmarks

def dead_reckon_se2(edges):
    poses = {0: (0.0, 0.0, 0.0)}
    odom = sorted([e for e in edges if e[1] == e[0] + 1], key=lambda e: e[0])
    for i, j, dx, dy, dth in odom:
        if i in poses:
            x0, y0, th0 = poses[i]
            c, s = math.cos(th0), math.sin(th0)
            poses[j] = (x0 + c*dx - s*dy, y0 + s*dx + c*dy, th0 + dth)
    return poses

def gt_from_se2_file(path):
    """Dead-reckon ground-truth trajectory from an SE2 ground-truth edge file."""
    edges, _ = parse_se2_file(path)
    return dead_reckon_se2(edges)

# ---- SE3 helpers ------------------------------------------------------------
def parse_se3_file(path):
    """Return (edges, point3_obs) from EDGE3/POINT3 file."""
    edges, point3 = [], {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            tok = parts[0]
            if tok == "EDGE3":
                edges.append({"from": int(parts[1]), "to": int(parts[2]),
                               "tx": float(parts[3]), "ty": float(parts[4]), "tz": float(parts[5]),
                               "rx": float(parts[6]), "ry": float(parts[7]), "rz": float(parts[8])})
            elif tok == "POINT3":
                lid = int(parts[2])
                if lid not in point3:
                    point3[lid] = (int(parts[1]), float(parts[3]), float(parts[4]), float(parts[5]))
    return edges, point3

def axis_angle_to_rot(rx, ry, rz):
    angle = math.sqrt(rx*rx + ry*ry + rz*rz)
    if angle < 1e-10:
        return np.eye(3)
    ax = np.array([rx, ry, rz]) / angle
    c, s = math.cos(angle), math.sin(angle)
    K = np.array([[0, -ax[2], ax[1]], [ax[2], 0, -ax[0]], [-ax[1], ax[0], 0]])
    return np.eye(3) + s * K + (1 - c) * (K @ K)

def dead_reckon_se3(edges, num_poses):
    R = [np.eye(3)] * num_poses
    t = [np.zeros(3)] * num_poses
    odom = sorted([e for e in edges if e["to"] == e["from"] + 1], key=lambda e: e["from"])
    for e in odom:
        i, j = e["from"], e["to"]
        if i < num_poses and j < num_poses:
            Rrel = axis_angle_to_rot(e["rx"], e["ry"], e["rz"])
            t[j] = t[i] + R[i] @ np.array([e["tx"], e["ty"], e["tz"]])
            R[j] = R[i] @ Rrel
    return t

def gt_from_se3_file(path):
    """Reconstruct ground-truth positions using BFS spanning tree over ALL edges.

    Uses every EDGE3 entry in the file (sequential + loop closures) to propagate
    poses from node 0 via breadth-first search.  Dead-reckoning on sequential
    edges alone accumulates large drift for sphere-topology datasets.
    """
    adj = defaultdict(list)   # node -> [(neighbor, Rrel, trel, is_inverse)]
    max_node = 0
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts or parts[0] != 'EDGE3':
                continue
            i, j = int(parts[1]), int(parts[2])
            max_node = max(max_node, i, j)
            tx, ty, tz = float(parts[3]), float(parts[4]), float(parts[5])
            rx, ry, rz = float(parts[6]), float(parts[7]), float(parts[8])
            Rrel = axis_angle_to_rot(rx, ry, rz)
            trel = np.array([tx, ty, tz])
            # Forward edge:  T_j = T_i  *  T_ij
            adj[i].append((j, Rrel, trel, False))
            # Inverse edge:  T_i = T_j  *  T_ij^{-1}
            adj[j].append((i, Rrel, trel, True))

    n = max_node + 1
    R_arr = [None] * n
    t_arr = [None] * n
    R_arr[0] = np.eye(3)
    t_arr[0] = np.zeros(3)

    visited = {0}
    queue = deque([0])
    while queue:
        parent = queue.popleft()
        for nb, Rrel, trel, inv in adj[parent]:
            if nb in visited:
                continue
            visited.add(nb)
            queue.append(nb)
            if not inv:
                # t_nb = t_p + R_p @ trel,   R_nb = R_p @ Rrel
                t_arr[nb] = t_arr[parent] + R_arr[parent] @ trel
                R_arr[nb] = R_arr[parent] @ Rrel
            else:
                # t_nb = t_p - R_p @ (Rrel^T @ trel),  R_nb = R_p @ Rrel^T
                t_arr[nb] = t_arr[parent] - R_arr[parent] @ (Rrel.T @ trel)
                R_arr[nb] = R_arr[parent] @ Rrel.T

    return t_arr   # list[np.ndarray], indexed by pose id

# ---- Plot helpers -----------------------------------------------------------
def set_axes_equal_3d(ax):
    """Enforce equal axis scales on a 3-D Axes3D.

    Matplotlib's 3-D backend ignores set_aspect('equal'), so we manually
    compute the tightest cube that contains all plotted data and apply it.
    """
    limits = np.array([ax.get_xlim3d(), ax.get_ylim3d(), ax.get_zlim3d()])
    center = limits.mean(axis=1)
    half   = (limits[:, 1] - limits[:, 0]).max() / 2.0
    ax.set_xlim3d([center[0] - half, center[0] + half])
    ax.set_ylim3d([center[1] - half, center[1] + half])
    ax.set_zlim3d([center[2] - half, center[2] + half])

print("Helpers loaded.")

Helpers loaded.


In [ ]:

# =============================================================================
# Section 3: Visualize ALL datasets
#   SE2  →  one figure with ground truth + optimised (+ landmarks overlaid)
#   SE3  →  two subplots: 3-D view (left) and top-down XY (right)
# =============================================================================

DATASETS = [
    # (json_stem, dataset_txt, groundtruth_txt_or_None, geometry)
    ("./build/city10000",          "city10000.txt",          "groundtruth/city10000_groundtruth.txt",          "SE2"),
    ("./build/cityTrees10000",     "cityTrees10000.txt",      "groundtruth/cityTrees10000_groundtruth.txt",     "SE2"),
    ("./build/manhattanOlson3500", "manhattanOlson3500.txt",  "groundtruth/manhattanOlson3500_groundtruth.txt", "SE2"),
    ("./build/sphere400",          "sphere400.txt",           None,                                             "SE3"),
    ("./build/sphere2500",         "sphere2500.txt",          "groundtruth/sphere2500_groundtruth.txt",         "SE3"),
    ("./build/torus10000",         "torus10000.txt",          None,                                             "SE3"),
    ("torus2000Points",    "torus2000Points.txt",     None,                                             "SE3"),
]

DATA_DIR  = os.path.join(REPO_ROOT, "data", "iSAM2")

for stem, dat_file, gt_file, geom in DATASETS:
    result_path = os.path.join(PROJ_ROOT, f"{stem}.json")
    if not os.path.exists(result_path):
        print(f"  [SKIP] {stem} — result JSON not found, run cell 1 first")
        continue

    with open(result_path) as f:
        res = json.load(f)

    dat_path = os.path.join(DATA_DIR, dat_file)
    gt_path  = os.path.join(DATA_DIR, gt_file) if gt_file else None

    # ------------------------------------------------------------------
    if geom == "SE2":
        edges_noisy, _ = parse_se2_file(dat_path)
        init_poses      = dead_reckon_se2(edges_noisy)

        # Ground truth
        gt_poses_xy = None
        if gt_path and os.path.exists(gt_path):
            gt_d    = gt_from_se2_file(gt_path)
            ids_gt  = sorted(gt_d.keys())
            gt_poses_xy = ([gt_d[i][0] for i in ids_gt], [gt_d[i][1] for i in ids_gt])

        # Initial (dead-reckoned from noisy data)
        ids_init = sorted(init_poses.keys())
        ix = [init_poses[i][0] for i in ids_init]
        iy = [init_poses[i][1] for i in ids_init]

        # Optimised
        opt_d   = {p["id"]: (p["x"], p["y"]) for p in res["poses"]}
        ids_opt = sorted(opt_d.keys())
        ox = [opt_d[i][0] for i in ids_opt]
        oy = [opt_d[i][1] for i in ids_opt]

        # Landmarks
        opt_lm = [(lm["x"], lm["y"]) for lm in res.get("landmarks", [])]

        fig, ax = plt.subplots(figsize=(10, 7))
        ax.plot(ix, iy, color='#aaaaaa', lw=0.6, label='Initial (dead-reckoned)')
        if gt_poses_xy:
            ax.plot(gt_poses_xy[0], gt_poses_xy[1],
                    color='#2ca02c', lw=1.0, label='Ground truth')
        ax.plot(ox, oy, color='#1f77b4', lw=1.0, label='Optimised')
        if opt_lm:
            ax.scatter([p[0] for p in opt_lm], [p[1] for p in opt_lm],
                       color='#ff7f0e', s=6, alpha=0.5, label='Landmarks', zorder=3)
        ax.set_xlabel("x (m)")
        ax.set_ylabel("y (m)")
        ax.set_aspect('equal')
        ax.legend(fontsize=9)
        ax.grid(True, lw=0.3, alpha=0.5)
        fig.suptitle(
            f"{stem}  [SE2]  poses={res['num_poses']}  lm={res['num_landmarks']}\n"
            f"error  {res['initial_error']:.3e} → {res['final_error']:.3e}   "
            f"iters={res['num_iterations']}",
            fontsize=10)
        plt.tight_layout()
        plt.show()

    # ------------------------------------------------------------------
    else:  # SE3
        edges3, _ = parse_se3_file(dat_path)
        num_p      = max((max(e["from"], e["to"]) for e in edges3), default=0) + 1
        init_t     = dead_reckon_se3(edges3, num_p)
        stride     = max(1, num_p // 3000)

        # Ground truth via BFS spanning tree
        gt_t = None
        if gt_path and os.path.exists(gt_path):
            gt_t = gt_from_se3_file(gt_path)

        # Optimised poses
        opt3   = {p["id"]: (p["x"], p["y"], p["z"]) for p in res["poses"]}
        ids_o  = sorted(opt3.keys())
        so     = max(1, len(ids_o) // 3000)
        ox3 = [opt3[i][0] for i in ids_o[::so]]
        oy3 = [opt3[i][1] for i in ids_o[::so]]
        oz3 = [opt3[i][2] for i in ids_o[::so]]

        # Subsampled initial
        ix3 = [init_t[i][0] for i in range(0, num_p, stride)]
        iy3 = [init_t[i][1] for i in range(0, num_p, stride)]
        iz3 = [init_t[i][2] for i in range(0, num_p, stride)]

        # Optional landmarks
        opt_lm3 = [(lm["x"], lm["y"], lm["z"]) for lm in res.get("landmarks", [])]

        fig = plt.figure(figsize=(18, 7))

        # --- 3-D view (left) ---
        ax3d = fig.add_subplot(1, 2, 1, projection='3d')
        ax3d.plot(ix3, iy3, iz3, color='#aaaaaa', lw=0.5, label='Initial')
        if gt_t is not None:
            sgt = max(1, len(gt_t) // 3000)
            ax3d.plot([gt_t[i][0] for i in range(0, len(gt_t), sgt)],
                      [gt_t[i][1] for i in range(0, len(gt_t), sgt)],
                      [gt_t[i][2] for i in range(0, len(gt_t), sgt)],
                      color='#2ca02c', lw=0.8, label='Ground truth')
        ax3d.plot(ox3, oy3, oz3, color='#1f77b4', lw=0.8, label='Optimised')
        if opt_lm3:
            ax3d.scatter([p[0] for p in opt_lm3], [p[1] for p in opt_lm3],
                         [p[2] for p in opt_lm3],
                         color='#ff7f0e', s=6, alpha=0.6, label='Landmarks')
        ax3d.set_xlabel("x"); ax3d.set_ylabel("y"); ax3d.set_zlabel("z")
        ax3d.legend(fontsize=8)
        ax3d.set_title("3-D view", fontsize=10)
        set_axes_equal_3d(ax3d)

        # --- Top-down XY (right) ---
        ax2d = fig.add_subplot(1, 2, 2)
        ax2d.plot(ix3, iy3, color='#aaaaaa', lw=0.5, label='Initial')
        if gt_t is not None:
            ax2d.plot([gt_t[i][0] for i in range(0, len(gt_t), sgt)],
                      [gt_t[i][1] for i in range(0, len(gt_t), sgt)],
                      color='#2ca02c', lw=0.8, label='Ground truth')
        ax2d.plot(ox3, oy3, color='#1f77b4', lw=0.8, label='Optimised')
        if opt_lm3:
            ax2d.scatter([p[0] for p in opt_lm3], [p[1] for p in opt_lm3],
                         color='#ff7f0e', s=6, alpha=0.6, label='Landmarks')
        ax2d.set_xlabel("x"); ax2d.set_ylabel("y")
        ax2d.set_aspect('equal')
        ax2d.legend(fontsize=8)
        ax2d.grid(True, lw=0.3, alpha=0.5)
        ax2d.set_title("Top-down XY", fontsize=10)

        fig.suptitle(
            f"{stem}  [SE3]  poses={res['num_poses']}  lm={res['num_landmarks']}\n"
            f"error  {res['initial_error']:.3e} → {res['final_error']:.3e}   "
            f"iters={res['num_iterations']}",
            fontsize=10)
        plt.tight_layout()
        plt.show()

print("All dataset plots rendered.")


  [SKIP] city10000 — result JSON not found, run cell 1 first
  [SKIP] cityTrees10000 — result JSON not found, run cell 1 first
  [SKIP] manhattanOlson3500 — result JSON not found, run cell 1 first
  [SKIP] sphere400 — result JSON not found, run cell 1 first
  [SKIP] sphere2500 — result JSON not found, run cell 1 first
  [SKIP] torus10000 — result JSON not found, run cell 1 first
  [SKIP] torus2000Points — result JSON not found, run cell 1 first
All dataset plots rendered.


In [4]:

# =============================================================================
# Section 4: Ground-truth error analysis + relation to factor-graph residuals
#
# Datasets with ground truth: city10000, cityTrees10000, manhattanOlson3500 (SE2)
#                             sphere2500 (SE3)
#
# Per-pose Euclidean error = ||optimised_position - gt_position||
# Both the optimizer and the BFS GT reconstruction are anchored at pose 0 = origin,
# so no additional alignment is needed.
#
# Factor-graph residual explanation
# ---------------------------------
# The stored final_error / initial_error are the total graph cost:
#
#   C = 0.5 * sum_f  r_f^T  Omega_f  r_f
#
# where r_f is the error residual for factor f and Omega_f is its information matrix.
# Omega = sigma^{-2} * I, so:
#   avg chi^2/factor < 1  means residuals are tighter than measurement noise.
#
# Typical information weights in the iSAM2 benchmark files:
#   SE2 translation dof: Omega ~ 50   (sigma_t ~ 1/sqrt(50) ~ 0.14 m)
#   SE3 translation dof: Omega ~ 10   (sigma_t ~ 1/sqrt(10) ~ 0.32 m)
# =============================================================================

import pandas as pd
import numpy as np
import math

GT_DATASETS = [
    ("city10000",          "groundtruth/city10000_groundtruth.txt",         "SE2"),
    ("cityTrees10000",     "groundtruth/cityTrees10000_groundtruth.txt",    "SE2"),
    ("manhattanOlson3500", "groundtruth/manhattanOlson3500_groundtruth.txt","SE2"),
    ("sphere2500",         "groundtruth/sphere2500_groundtruth.txt",        "SE3"),
]

# Typical translation information diagonal used in iSAM2 benchmark data
OMEGA_TRANS = {"SE2": 50.0, "SE3": 10.0}

rows = []

for stem, gt_rel, geom in GT_DATASETS:
    res     = RESULTS[stem]
    gt_path = os.path.join(DATA_DIR, gt_rel)

    # ---- Reconstruct ground-truth positions ---------------------------------
    if geom == "SE2":
        gt_d   = gt_from_se2_file(gt_path)
        opt_d  = {p["id"]: (p["x"], p["y"]) for p in res["poses"]}
        ids    = sorted(set(gt_d.keys()) & set(opt_d.keys()))
        gt_xy  = np.array([(gt_d[i][0],  gt_d[i][1])  for i in ids])
        opt_xy = np.array([(opt_d[i][0], opt_d[i][1]) for i in ids])
        errors = np.linalg.norm(opt_xy - gt_xy, axis=1)
    else:  # SE3 — use BFS spanning-tree GT
        gt_t   = gt_from_se3_file(gt_path)
        opt_d  = {p["id"]: (p["x"], p["y"], p["z"]) for p in res["poses"]}
        ids    = sorted(opt_d.keys())
        gt_xyz  = np.array([gt_t[i]   for i in ids])
        opt_xyz = np.array([opt_d[i]  for i in ids])
        errors  = np.linalg.norm(opt_xyz - gt_xyz, axis=1)

    n_f  = res["num_factors"]
    fe   = res["final_error"]
    ie   = res["initial_error"]
    rmse = math.sqrt(np.mean(errors**2))

    # Average chi^2 per factor at convergence
    avg_chi2 = fe / n_f

    # Back-computed per-DOF RMS residual from the factor graph cost.
    # C_avg = 0.5 * Omega * r^2  =>  r_rms = sqrt(2 * C_avg / Omega)
    omega    = OMEGA_TRANS[geom]
    infer_rms = math.sqrt(2.0 * avg_chi2 / omega)

    rows.append({
        "Dataset":             stem,
        "Geom":                geom,
        "N poses":             res["num_poses"],
        "N factors":           n_f,
        "Initial error":       f"{ie:.3e}",
        "Final error":         f"{fe:.3e}",
        "Reduction (%)":       f"{(ie - fe) / ie * 100:.1f}",
        "Error min (m)":       round(float(errors.min()),       4),
        "Error max (m)":       round(float(errors.max()),       4),
        "Error mean (m)":      round(float(errors.mean()),      4),
        "Error median (m)":    round(float(np.median(errors)),  4),
        "Error std (m)":       round(float(errors.std()),       4),
        "Error RMSE (m)":      round(rmse,                      4),
        "chi2 / factor":       round(avg_chi2,                  4),
        "Inferred sigma_t (m)":round(infer_rms,                 4),
    })

df = pd.DataFrame(rows).set_index("Dataset")
pd.set_option("display.max_colwidth", None)

print("── Optimisation summary ──────────────────────────────────────────────────────")
display(df[["Geom", "N poses", "N factors", "Initial error", "Final error", "Reduction (%)"]])

print("\n── Per-pose Euclidean GT errors (metres) ─────────────────────────────────────")
err_cols = ["Geom", "Error min (m)", "Error max (m)", "Error mean (m)",
            "Error median (m)", "Error std (m)", "Error RMSE (m)"]
display(
    df[err_cols].style
      .background_gradient(subset=["Error RMSE (m)"], cmap="YlOrRd")
      .format("{:.4f}", subset=[c for c in err_cols if c != "Geom"])
)

print("\n── Factor-graph residuals vs. GT errors ──────────────────────────────────────")
res_cols = ["Geom", "chi2 / factor", "Inferred sigma_t (m)", "Error RMSE (m)"]
display(
    df[res_cols].style
      .background_gradient(subset=["chi2 / factor"], cmap="Blues")
      .format("{:.4f}", subset=[c for c in res_cols if c != "Geom"])
)

── Optimisation summary ──────────────────────────────────────────────────────


,Geom,N poses,N factors,Initial error,Final error,Reduction (%)
Dataset,,,,,,
city10000,SE2,10000,20688,3.271e+08,2.560e+02,100.0
cityTrees10000,SE2,10000,14443,5.154e+05,2.723e+02,99.9
manhattanOlson3500,SE2,3500,5599,1.283e+06,7.304e+01,100.0
sphere2500,SE3,2500,4950,1.596e+06,3.651e+02,100.0



── Per-pose Euclidean GT errors (metres) ─────────────────────────────────────


,Geom,Error min (m),Error max (m),Error mean (m),Error median (m),Error std (m),Error RMSE (m)
Dataset,,,,,,,
city10000,SE2,0.0000,0.2166,0.0481,0.0400,0.0328,0.0582
cityTrees10000,SE2,0.0000,1.6110,0.7185,0.7572,0.2575,0.7632
manhattanOlson3500,SE2,0.0000,4.2466,0.8021,0.6080,0.8652,1.1798
sphere2500,SE3,0.0000,3.1118,1.7467,1.9014,0.8303,1.9340



── Factor-graph residuals vs. GT errors ──────────────────────────────────────


,Geom,chi2 / factor,Inferred sigma_t (m),Error RMSE (m)
Dataset,,,,
city10000,SE2,0.0124,0.0222,0.0582
cityTrees10000,SE2,0.0189,0.0275,0.7632
manhattanOlson3500,SE2,0.0130,0.0228,1.1798
sphere2500,SE3,0.0738,0.1215,1.9340
